# Atividade Prática — Aula 6: Visualização Interativa com Plotly

Esta atividade foi construída com base nos slides da Aula 6, que apresentam a transição do gráfico estático para o gráfico como **aplicativo de exploração**, com foco em **hover**, **zoom**, **pan**, **filtros visuais**, **Plotly Express** e construção de componentes que mais tarde podem virar um dashboard. fileciteturn7file0

## Ideia central da aula
A interatividade existe para apoiar a investigação do gestor em tempo real.  
Mesmo assim, a aula reforça uma regra essencial: **interatividade não substitui clareza**. O gráfico precisa continuar limpo, bem titulado e orientado à decisão. fileciteturn7file0

## Regras da atividade
- O notebook orienta, mas **você deve construir os códigos**.
- Use **Plotly Express** sempre que possível.
- Após cada visual principal, escreva uma breve **interpretação humana**.
- Teste hover, zoom e isolamento por legenda antes de concluir.
- Pense como desenvolvedor: os gráficos de hoje poderão virar o dashboard de amanhã. fileciteturn7file0

## Dataset da atividade
Arquivo: `vendas_brasil_clean_aula6_plotly.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias.

**Sugestão:**
- `pandas`
- `plotly.express`
- `plotly.graph_objects` (opcional)


In [2]:
# Escreva aqui suas importações
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 2. Leitura e inspeção inicial da base

Leia o arquivo `vendas_brasil_clean_aula6_plotly.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. verifique o tamanho da base
3. confira os tipos das colunas
4. confirme se `data_venda` está em formato adequado para análises temporais
5. identifique quais colunas podem alimentar:
   - comparação de categorias
   - evolução no tempo
   - relação entre variáveis
   - distribuição espacial


In [3]:
# Leia o CSV e faça a inspeção inicial
df = pd.read_csv("../data/vendas_brasil_clean_aula6_plotly.csv")

primeiras_linhas = df.head()
shape_df = df.shape
tipos_colunas = df.dtypes

print(primeiras_linhas)
print()
print("Shape:", shape_df)
print()
print("Tipos das colunas:")
print(tipos_colunas)

df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

colunas_comparacao = ["canal_venda", "categoria", "uf", "produto"]
colunas_tempo = ["data_venda"]
colunas_relacao = ["receita", "lucro", "margem_lucro", "quantidade"]
colunas_espacial = ["latitude", "longitude", "uf"]

print()
print("Colunas para comparação:", colunas_comparacao)
print("Colunas para tempo:", colunas_tempo)
print("Colunas para relação:", colunas_relacao)
print("Colunas para espacial:", colunas_espacial)

df.info()

   pedido_id  data_venda  uf  canal_venda    categoria           produto  \
0       9000  2025-01-29  RS  Marketplace   Acessórios  Teclado Mecânico   
1       9001  2025-03-10  PE          App       Móveis     Mesa Compacta   
2       9002  2025-03-25  SC       Online  Informática        Monitor 27   
3       9003  2025-06-20  BA       Online       Móveis    Cadeira Office   
4       9004  2025-07-19  RS       Online    Telefonia      Smartphone X   

   quantidade  desconto  preco_unitario  receita    lucro  margem_lucro  \
0           5    0.0700          386.96  4063.08   675.62        0.1663   
1           3    0.1265          614.56  1843.68   271.35        0.1472   
2           6    0.0712         1339.74  8038.44  2011.38        0.2502   
3           4    0.0000         1004.15  4016.60  1326.00        0.3301   
4           2    0.0000         2512.55  5025.10   762.04        0.1516   

   latitude  longitude      mes  
0 -29.86937  -50.89149  2025-01  
1  -8.58550  -35.20490  

## 3. Matriz de decisão do analista

Os slides apresentam uma regra prática: usar interativo (Plotly) quando o painel será visto em tela/web, quando o gestor precisará responder subperguntas na hora e quando o gráfico poderá ser empacotado em um dashboard. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Por que esta atividade faz mais sentido em Plotly do que em Matplotlib?
2. Em que situação você ainda preferiria um gráfico estático?
3. O destino desta análise parece mais “PDF” ou mais “tela/web”?


Esta atividade faz mais sentido em Plotly porque o objetivo não é apenas mostrar um gráfico pronto, mas permitir exploração em tempo real com hover, zoom, pan e isolamento visual por legenda. Isso ajuda o gestor a investigar subperguntas durante a análise.

Eu ainda preferiria um gráfico estático quando o destino final fosse um relatório em PDF, um trabalho acadêmico impresso ou uma apresentação em que a mensagem já estivesse totalmente fechada e não precisasse de exploração.

O destino desta análise parece claramente mais orientado a tela ou web, porque os recursos interativos fazem mais sentido em ambiente digital e podem futuramente virar componentes de dashboard.

## 4. Missão 1 — Barras: Receita por Canal

A missão prática da aula pede construir um gráfico de barras para responder:
**qual canal vende mais?**  
Os slides também sugerem usar o hover para injetar métricas secundárias sem poluir a tela. fileciteturn7file0

### Tarefa
1. Agregue a `receita` por `canal_venda`
2. Inclua também uma métrica secundária, como `quantidade`
3. Construa um gráfico de barras com Plotly Express
4. Ordene do maior para o menor
5. Faça o hover mostrar mais do que apenas a receita

### Perguntas
- Qual canal lidera a receita?
- O canal líder também lidera em quantidade?
- O hover ajudou a enriquecer a leitura sem poluir a tela?


In [4]:
# Construa aqui o gráfico de barras interativo por canal_venda
df_canal = (
    df.groupby("canal_venda", as_index=False)
    .agg(
        receita_total=("receita", "sum"),
        quantidade_total=("quantidade", "sum"),
        lucro_total=("lucro", "sum")
    )
    .sort_values("receita_total", ascending=False)
)

fig_canal = px.bar(
    df_canal,
    x="canal_venda",
    y="receita_total",
    hover_data={
        "quantidade_total": True,
        "lucro_total": True,
        "receita_total": ":,.2f"
    },
    title="Receita total por canal de venda",
    labels={
        "canal_venda": "Canal de venda",
        "receita_total": "Receita total"
    }
)

fig_canal.update_layout(xaxis_title="Canal de venda", yaxis_title="Receita total")
fig_canal.show()

### Insight obrigatório
Escreva 2 ou 3 linhas explicando:
- quem lidera
- quem fica atrás
- o que um gestor poderia investigar a seguir


## 5. Missão 2 — Linhas: Sazonalidade da Receita Mensal

Os slides destacam que o gráfico de linhas, com zoom e pan, é ideal para navegar no tempo e isolar picos. Também reforçam que o eixo temporal precisa estar corretamente tipado no Pandas. fileciteturn7file0

### Tarefa
1. Converta `data_venda` em data, se necessário
2. Agregue a `receita` por mês
3. Construa um gráfico de linha interativo
4. Teste o zoom nos meses de pico
5. Observe se novembro e dezembro se destacam

### Perguntas
- Existe sazonalidade?
- Quais meses chamam atenção?
- O zoom ajudou a explorar melhor a parte final da série?


In [5]:
# Construa aqui o gráfico de linha interativo da receita mensal
df_tempo = df.copy()
df_tempo["data_venda"] = pd.to_datetime(df_tempo["data_venda"], errors="coerce")
df_tempo["mes"] = df_tempo["data_venda"].dt.to_period("M").astype(str)

df_mensal = (
    df_tempo.groupby("mes", as_index=False)
    .agg(receita_total=("receita", "sum"))
    .sort_values("mes")
)

fig_mensal = px.line(
    df_mensal,
    x="mes",
    y="receita_total",
    markers=True,
    title="Sazonalidade da receita mensal",
    labels={
        "mes": "Mês",
        "receita_total": "Receita total"
    }
)

fig_mensal.update_layout(xaxis_title="Mês", yaxis_title="Receita total")
fig_mensal.show()

### Insight obrigatório
Explique:
- qual tendência aparece
- se há picos claros
- qual hipótese de negócio pode explicar o comportamento observado


## 6. Missão 3 — Dispersão: Lucro vs. Receita segmentado por Categoria

Os slides mostram o uso do gráfico de dispersão para revelar correlação entre KPIs e identificar anomalias, com grande apoio do hover. fileciteturn7file0

### Tarefa
1. Construa um scatter plot com:
   - eixo X = `receita`
   - eixo Y = `lucro`
2. Use `categoria` como cor
3. Inclua no hover:
   - `produto`
   - `canal_venda`
   - `uf`
   - `margem_lucro`
4. Tente identificar algum ponto fora da curva

### Perguntas
- Existe correlação visual entre receita e lucro?
- Há anomalias?
- O hover ajuda a transformar um ponto em um caso investigável?


In [6]:
# Construa aqui o scatter plot interativo entre receita e lucro
fig_scatter = px.scatter(
    df,
    x="receita",
    y="lucro",
    color="categoria",
    hover_data={
        "produto": True,
        "canal_venda": True,
        "uf": True,
        "margem_lucro": ":.2%"
    },
    title="Relação entre receita e lucro por categoria",
    labels={
        "receita": "Receita",
        "lucro": "Lucro",
        "categoria": "Categoria"
    }
)

fig_scatter.update_layout(xaxis_title="Receita", yaxis_title="Lucro")
fig_scatter.show()

### Insight obrigatório
Escreva 2 ou 3 linhas dizendo:
- se a correlação parece positiva
- onde surgem pontos fora do padrão
- que ação analítica poderia ser tomada depois


## 7. Exploração espacial — Onde está concentrada a operação?

Os slides incluem mapas geográficos como resposta para perguntas espaciais como:
**onde está concentrada nossa operação?** fileciteturn7file0

### Tarefa
Usando `latitude` e `longitude`, crie um mapa interativo que ajude a visualizar a operação.

### Sugestões
- use tamanho ou cor para uma métrica relevante (como receita)
- teste o zoom do mapa
- observe se há concentração regional

### Perguntas
- Onde a operação parece mais concentrada?
- O mapa ajudou mais do que uma simples tabela por UF?


In [9]:
# Construa aqui um mapa interativo com Plotly
fig_mapa = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    size="receita",
    color="lucro",
    hover_name="produto",
    hover_data={
        "uf": True,
        "canal_venda": True,
        "receita": ":,.2f",
        "lucro": ":,.2f"
    },
    zoom=3,
    height=600,
    title="Concentração geográfica da operação"
)

fig_mapa.update_layout(mapbox_style="open-street-map")
fig_mapa.show()

## 8. A tríade da interatividade

Um slide central da aula apresenta três componentes principais:
- **Tooltips (hover)**
- **Zoom & Pan**
- **Filtros visuais / isolamento via legenda** fileciteturn7file0

### Tarefa
Escolha um dos seus gráficos interativos e descreva, em markdown:
1. O que o hover acrescenta
2. Como o zoom melhora a investigação
3. Como o clique na legenda pode ajudar a isolar séries ou categorias



```
No gráfico de dispersão, o hover acrescenta contexto detalhado sem poluir a visualização principal, permitindo identificar produto, canal, UF e margem de cada ponto. O zoom ajuda a investigar regiões mais densas ou áreas com possíveis anomalias. O clique na legenda permite isolar categorias específicas, facilitando comparações e reduzindo ruído visual.
```


## 9. Plotly Express — Máximo impacto, mínimo código

A aula mostra o Plotly Express como uma API declarativa, perfeita para DataFrames do Pandas e com geração automática de HTML/JS. fileciteturn7file0

### Tarefa
Explique em markdown:
1. O que significa dizer que Plotly Express é “declarativo”?
2. Em que ele facilita a vida do analista?
3. Como isso se conecta com a ideia de futuro dashboard em Streamlit?


```
Dizer que Plotly Express é declarativo significa que o analista informa quais colunas devem ocupar eixos, cor, tamanho e hover, e a biblioteca monta automaticamente a estrutura visual. Isso acelera muito o trabalho, reduz código repetitivo e integra naturalmente com DataFrames do Pandas. Essa abordagem facilita o reaproveitamento em aplicações como Streamlit, porque os gráficos já nascem como componentes prontos para ambiente web.
```

## 10. Clareza continua obrigatória

A aula reforça que um gráfico interativo ruim apenas confunde o usuário de forma mais tecnológica.  
Os princípios inegociáveis continuam sendo:
- título autoexplicativo
- eixos nomeados
- unidades claras
- remoção de lixo visual fileciteturn7file0

### Tarefa
Revise um dos seus gráficos e melhore:
- título
- rótulos de eixo
- hover
- nomes das variáveis
- aparência geral

Depois escreva:
1. O que você mudou?
2. O gráfico ficou mais claro?


In [10]:
# Refaça aqui um dos gráficos com foco em clareza
df_canal_claro = (
    df.groupby("canal_venda", as_index=False)
    .agg(
        receita_total=("receita", "sum"),
        quantidade_total=("quantidade", "sum"),
        lucro_total=("lucro", "sum")
    )
    .sort_values("receita_total", ascending=False)
)

fig_canal_claro = px.bar(
    df_canal_claro,
    x="canal_venda",
    y="receita_total",
    text="receita_total",
    hover_data={
        "quantidade_total": True,
        "lucro_total": ":,.2f",
        "receita_total": ":,.2f"
    },
    title="Canal líder concentra a maior geração de receita",
    labels={
        "canal_venda": "Canal de venda",
        "receita_total": "Receita total",
        "quantidade_total": "Quantidade total",
        "lucro_total": "Lucro total"
    }
)

fig_canal_claro.update_traces(texttemplate="%{text:,.2f}", textposition="outside")
fig_canal_claro.update_layout(xaxis_title="Canal de venda", yaxis_title="Receita total")
fig_canal_claro.show()

## 11. O gráfico não fala sozinho

O último slide deixa explícito: mesmo com zoom e tooltips avançados, a interpretação humana continua insubstituível. Sempre forneça uma conclusão textual acompanhando o visual. fileciteturn7file0

### Tarefa
Escolha **dois gráficos** que você construiu e, para cada um, escreva:
- um insight principal
- uma possível decisão gerencial
- uma pergunta adicional que o gestor poderia fazer em seguida


### Gráfico de barras por canal
- Insight principal: um canal concentra a maior parte da receita total da operação.
- Possível decisão gerencial: priorizar investimento e otimização comercial nesse canal, sem perder de vista sua margem.
- Pergunta seguinte: esse canal também lidera em lucro e eficiência operacional?

### Gráfico de linha mensal
- Insight principal: a receita apresenta meses mais fortes, sugerindo sazonalidade.
- Possível decisão gerencial: preparar campanhas, estoque e operação para os períodos de pico.
- Pergunta seguinte: o crescimento mensal também se traduz em maior lucro ou apenas em maior volume?

## 12. Missão prática final — Mapeando e construindo

Com base no quadro de missão prática da aula, entregue no notebook, no mínimo: fileciteturn7file0

1. **Barras** — Receita por canal, com hover enriquecido
2. **Linhas** — Sazonalidade da receita mensal, explorando zoom
3. **Dispersão** — Lucro vs. Receita segmentado por categoria, com hover e identificação de anomalia

### Extra recomendado
4. **Mapa** — concentração geográfica da operação

### Para cada gráfico
Inclua abaixo:
- a pergunta de negócio
- as colunas necessárias
- a interpretação humana final


#
- Pergunta de negócio: qual canal vende mais?
- Colunas necessárias: canal_venda, receita, quantidade, lucro
- Interpretação humana: o ranking mostra quais canais concentram o faturamento e abre espaço para investigar eficiência por canal.

#
- Pergunta de negócio: como a receita varia ao longo do tempo?
- Colunas necessárias: data_venda, receita
- Interpretação humana: a série temporal permite observar tendência, sazonalidade e meses de pico.

#
- Pergunta de negócio: receita alta gera lucro alto em todos os casos?
- Colunas necessárias: receita, lucro, categoria, produto, canal_venda, uf, margem_lucro
- Interpretação humana: a nuvem de pontos ajuda a visualizar correlação e detectar anomalias que merecem investigação.

#
- Pergunta de negócio: onde está concentrada a operação?
- Colunas necessárias: latitude, longitude, receita, lucro, uf
- Interpretação humana: o mapa mostra concentração geográfica de forma mais intuitiva do que uma tabela isolada.

## 13. Mindset de desenvolvedor

Um dos slides diz que os gráficos de hoje são o dashboard de amanhã.  
Isso exige padronização desde já: nomes coerentes, lógica limpa e reaproveitamento futuro. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Como você nomearia seus objetos `fig_...` de forma organizada?
2. Que partes do seu código poderiam ser reaproveitadas numa aplicação web?
3. O que vale a pena padronizar desde esta aula?


```
Eu nomearia os objetos de forma descritiva, como `fig_receita_canal`, `fig_receita_mensal`, `fig_lucro_receita_categoria` e `fig_mapa_operacao`. O que pode ser reaproveitado numa aplicação web é a lógica de agregação, os DataFrames intermediários e os próprios objetos de figura. Vale a pena padronizar nomes de variáveis, estrutura das agregações, convenções de título, labels dos eixos e campos de hover desde esta aula.
```

## 14. Desafio extra (opcional)

Crie mais um gráfico interativo à sua escolha, desde que ele responda uma pergunta real. Exemplos:
- receita por UF (barras)
- preço unitário ao longo do tempo (linhas)
- desconto vs. quantidade (dispersão)
- receita por categoria (barras horizontais)

Mas atenção:
- use hover com intenção
- teste zoom se fizer sentido
- escreva interpretação final


In [11]:
# Desafio extra opcional
df_uf = (
    df.groupby("uf", as_index=False)
    .agg(
        receita_total=("receita", "sum"),
        lucro_total=("lucro", "sum"),
        quantidade_total=("quantidade", "sum")
    )
    .sort_values("receita_total", ascending=False)
)

fig_uf = px.bar(
    df_uf,
    x="uf",
    y="receita_total",
    hover_data={
        "lucro_total": ":,.2f",
        "quantidade_total": True,
        "receita_total": ":,.2f"
    },
    title="Receita total por UF",
    labels={
        "uf": "UF",
        "receita_total": "Receita total"
    }
)

fig_uf.update_layout(xaxis_title="UF", yaxis_title="Receita total")
fig_uf.show()

## 15. Entrega esperada

Seu notebook deve demonstrar:
- uso correto de Plotly Express
- escolha adequada do gráfico para a pergunta
- exploração consciente de hover, zoom, pan e legenda
- compromisso com clareza
- interpretação textual consistente

### Mensagem principal da aula
A tecnologia explora.  
Mas só o analista transforma exploração em decisão. fileciteturn7file0
